In [ ]:
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.document_loaders.csv_loader import CSVLoader
from langchain_community.vectorstores import FAISS
import pandas as pd
from langchain_classic.chains import RetrievalQA
from langchain_classic.agents import initialize_agent, AgentType, Tool
from langchain_core.documents import Document
import openai
from datetime import datetime
import re
from datetime import time as time_class
# ---------------------------------------------------------------------------
# Unified experiment configuration + run logging  (see PATCH_GUIDE.md)
# ---------------------------------------------------------------------------
import os
import leanopt_exp as lx

# API keys come from a .env file at the repository root (see .env.example),
# from the shell environment, or -- if neither is set -- from a hidden prompt.
# Nothing is hard-coded here and nothing is written to the run log.
try:
    from dotenv import load_dotenv
    # override=True: the .env file wins over anything already in the
    # environment. Without it, a stale key left in os.environ by an earlier
    # run silently shadows the value you just edited into .env.
    load_dotenv(override=False)
except ImportError:
    pass

CFG = lx.load_config(
    "exp_config.yaml",
    model_profile='gpt-4.1',
    method='LEAN-LLM-OPT',
    dataset='Air-NRM-NP',
    repeat_index=0,
)
lx.ensure_api_keys(CFG)                          # only for providers this profile uses
user_api_key = os.environ.get("OPENAI_API_KEY", "")   # legacy references

LOG = lx.RunLogger(CFG)
EMBEDDINGS = lx.build_embeddings(CFG)

# Problem Classification

In [ ]:
def Classification_Agent(file_path="Large_Scale_Or_Files/RefData.csv"):
    # Initialize the LLM
    llm1 = lx.build_llm(CFG, "classifier")

    # Load and process the data
    # classification only needs the problem description + its class label
    refdata = lx.load_refdata_docs(CFG)
    refdocuments = refdata

    embeddings = EMBEDDINGS
    vectors1 = FAISS.from_documents(refdocuments, embeddings)
    retriever1 = lx.build_retriever(CFG, vectors1, "refdata")
    qa_chain = RetrievalQA.from_chain_type(
        llm=llm1,
        chain_type="stuff",
        retriever=retriever1,
        return_source_documents=True,
    )

    qa_tool1 = Tool(
        name="FileQA",
        func=qa_chain.invoke,
        description=(
            "Use this tool to answer questions about the problem type of the text. "
            "Provide the question as input, and the tool will retrieve the relevant information from the file and use it to answer the question."
        ),
    )

    few_shot_examples = """

    Question: What is the problem type in operation of the text? Please give the answer directly. Text:There are three best-selling items (P1, P2, P3) on Amazon with the profit w_1,w_2,w_3.There is an independent demand stream for each of the products. The objective of the company is to decide which demands to be fufilled over a ﬁnite sales horizon [0,10] to maximize the total expected revenue from ﬁxed initial inventories. The on-hand inventories for the three items are c_1,c_2,c_3 respectively. During the sales horizon, replenishment is not allowed and there is no any in-transit inventories. Customers who want to purchase P1,P2,P3 arrive at each period accoring to a Poisson process with a_1,a_2,a_3 the arrival rates respectively. Decision variables y_1,y_2,y_3 correspond to the number of requests that the firm plans to fulfill for product 1,2,3. These variables are all positive integers.

    Thought: I need to determine the problem type of the text. I'll use the FileQA tool to retrieve the relevant information.

    Action: FileQA

    Action Input: "What is the problem type in operation of the text? text:There are three best-selling items (P1, P2, P3) on Amazon with the profit w_1, w_2, w_3. ..."

    Observation: The problem type of the text is Network Revenue Management.

    Final Answer: Network Revenue Management.

    """

    prefix = f"""You are a helpful assistant that can answer questions about operation problems.

    Use the following examples as a guide. Always use the FileQA tool when you need to retrieve information from the file:


    {few_shot_examples}

    When you need to find information from the file, use the provided tools. And answer the question by given the answer directly. For example,

    """

    suffix = """

    Begin!

    Question: {input}
    {agent_scratchpad}"""

    agent_pc = initialize_agent(
        tools=[qa_tool1],
        llm=llm1,
        agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
        **lx.agent_kwargs(CFG, prefix.format(few_shot_examples=few_shot_examples), suffix),
    )

    openai.api_request_timeout = 60 
    return agent_pc

def Problemtype(query):
    agent_pc = Classification_Agent(file_path="Large_Scale_Or_Files/RefData.csv")
    category_original=agent_pc.invoke(f"What is the problem type in operation of the text? text:{query}")
    type_output = category_original['output']
    return type_output

# ProblemSolvingModel

## PreProcess: Sharing Functions

In [ ]:
def LoadFiles():
  v1 = pd.read_csv('Test_Dataset/Air_NRM/small_scale/v1.csv')
  v2 = pd.read_csv('Test_Dataset/Air_NRM/small_scale/v2.csv')
  demand = pd.read_csv('Test_Dataset/Air_NRM/small_scale/od_demand.csv')
  flight = pd.read_csv('Test_Dataset/Air_NRM/small_scale/flight.csv')
  return v1,v2,demand,flight

In [ ]:
def New_Vectors_Flight(query):
  v1,v2,demand,df = LoadFiles()
  new_docs = []
  for _, row in df.iterrows():
      new_docs.append(Document(
          page_content=f"OD={row['Oneway_OD']},Departure_Time_Flight1={row['Departure Time']},Oneway_Product={row['Oneway_Product']},avg_price={row['Avg Price']}",
          metadata={
              'OD': row['Oneway_OD'],
              'time': row['Departure Time'],
              'product': row['Oneway_Product'],
              'avg_price': row['Avg Price']
          }
      ))

  embeddings = EMBEDDINGS
  new_vectors = FAISS.from_documents(new_docs, embeddings)
  return new_vectors

def New_Vectors_Demand(query):
  v1,v2,demand,df = LoadFiles()
  new_docs = []
  for _, row in demand.iterrows():
      new_docs.append(Document(
          page_content=f"OD={row['Oneway_OD']}, avg_pax={row['Avg Pax']}",
          metadata={
              'OD': row['Oneway_OD'],
              'avg_pax': row['Avg Pax']
          }
      ))

  embeddings = EMBEDDINGS
  new_vectors = FAISS.from_documents(new_docs, embeddings)
  return new_vectors


def retrieve_key_information(query):
    flight_pattern = r"\(OD\s*=\s*\(\s*'(\w+)'\s*,\s*'(\w+)'\s*\)\s+AND\s+Departure\s+Time='(\d{1,2}:\d{2})'\)"
    matches = re.findall(flight_pattern, query)

    flight_strings = [f"(OD = ('{origin}', '{destination}') AND Departure Time='{departure_time}')" for origin, destination, departure_time in matches]

    new_query = "Retrieve avg_price, avg_pax, value_list, ratio_list, value_0_list, ratio_0_list, and flight_capacity for the following flights:\n" + ", ".join(flight_strings) + "."

    return new_query
def retrieve_time_period(departure_time):
    intervals = {
        '12pm~6pm': (time_class(12, 0), time_class(18, 0)),
        '6pm~10pm': (time_class(18, 0), time_class(22, 0)),
        '10pm~8am': (time_class(22, 0), time_class(8, 0)),
        '8am~12pm': (time_class(8, 0), time_class(12, 0))
    }

    if isinstance(departure_time, str):
        try:
            hours, minutes = map(int, departure_time.split(':'))
            departure_time = time_class(hours, minutes)
        except ValueError:
            raise ValueError("Time format should be: 'HH:MM'")

    for interval_name, (start, end) in intervals.items():
        if start < end:
            if start <= departure_time < end:
                return interval_name
        else:
            if departure_time >= start or departure_time < end:
                return interval_name

    return "Unknown"
def retrieve_parameter(O,time_interval,product):
    v1,v2,df,demand = LoadFiles()
    time_interval = f'({time_interval})'
    key = product + '*' + time_interval
    _value_ = 0
    _ratio_ = 0
    no_purchase_value = 0
    no_purchase_value_ratio = 0
    subset = v1[v1['OD Pairs'] == O]
    if key in subset.columns and not subset.empty:
        _value_ = subset[key].values[0]

    subset2 = v2[v2['OD Pairs'] == O]
    if key in subset2.columns and not subset2.empty:
        _ratio_ = subset2[key].values[0]

    if 'no_purchase' in subset.columns and not subset.empty:
        no_purchase_value = subset['no_purchase'].values[0]

    if 'no_purchase' in subset2.columns and not subset2.empty:
        no_purchase_value_ratio = subset2['no_purchase'].values[0]
    return _value_,_ratio_,no_purchase_value,no_purchase_value_ratio

def generate_coefficients(OD,time):
    departure_time = datetime.strptime(time, '%H:%M').time()
    time_interval = retrieve_time_period(departure_time)
    value_1,ratio_1,value_0,ratio_0 = retrieve_parameter(OD,time_interval,'Eco_flexi')

    value_2,ratio_2,value_0,ratio_0 = retrieve_parameter(OD,time_interval,'Eco_lite')

    return value_1,ratio_1,value_2,ratio_2,value_0,ratio_0


def clean_text_preserve_newlines(text):
    cleaned = re.sub(r'\x1b\[[0-9;]*[mK]', '', text)
    cleaned = re.sub(r'[^\x20-\x7E\n]', '', cleaned)
    cleaned = re.sub(r'(\n\s+)(\w+\s*=)', r'\n\2', cleaned)
    cleaned = re.sub(r'\[\s+', '[', cleaned)
    cleaned = re.sub(r'\s+\]', ']', cleaned)
    cleaned = re.sub(r',\s+', ', ', cleaned)

    return cleaned


## Flow

In [ ]:
# SMALL-SCALE NP/FLOW: retriever-backed data tools and stable integer keys.
from Test_Dataset.Air_NRM.small_scale.np_lean_workflow import install_np_workflow

_NP_WORKFLOW = install_np_workflow(globals())
globals().update(_NP_WORKFLOW)


## CA

In [ ]:
# CA workflow omitted from this NP-only notebook.


# Combined Function

In [ ]:
def Process_Input(query):
    with lx.stage("classification"):
        category_original = Problemtype(query)
    print(f"Problem type classification finished: {category_original}.")
    if "Sales-Based Linear Programming" not in category_original:
        raise ValueError(f"unroutable problem type: {category_original!r}")
    if "flow conservation constraint" not in query.lower():
        raise ValueError("this NP-only notebook requires a flow-conservation query")
    with lx.stage("modeling"):
        output_model = ProcessPolicyFlow(query)
    return "Policy_Flow", output_model


# Run the test

In [ ]:
import re as _re
from tqdm import tqdm


_FENCE_RE = _re.compile(r"```(?:python|py)\s*\n(.*?)```", _re.I | _re.S)


def _extract_gurobipy_code(text):
    blocks = _FENCE_RE.findall(str(text or ""))
    if len(blocks) != 1 or "gurobipy" not in blocks[0]:
        raise ValueError("expected one complete gurobipy code block")
    return blocks[0].strip()


def Batch_Process_Queries(df, query_column="Query", log=LOG, cfg=CFG,
                          gold_column="Label-objective"):
    results = []
    has_gold = gold_column in df.columns
    for pos, (idx, row) in enumerate(tqdm(list(df.iterrows()), desc="Processing Queries")):
        query = row[query_column]
        instance_id = int(idx) if isinstance(idx, int) else pos
        gold = row[gold_column] if has_gold else None
        category, raw_model_output = None, None
        with log.instance(instance_id=instance_id, query=query,
                          gold_objective=gold) as rec:
            try:
                NP_SET_CURRENT_QUERY(query)
                category, raw_model_output = Process_Input(query)
                executable_code = _extract_gurobipy_code(raw_model_output)
                rec.set(pred_type=category, model_output=raw_model_output,
                        code_output=executable_code,
                        code_assembly="validated_direct_LLM_code")
            except Exception as exc:
                print(f"[{instance_id}] failed: {type(exc).__name__}: {exc}")
                rec.fail(exc)
        results.append({"Category": category, "Original_Query": query,
                        "Output": raw_model_output})
    return pd.DataFrame(results)


In [ ]:
# Generated code is scored after the run.


### Run small-scale NP / Flow


In [ ]:
# Small-scale NP-only full run.
df = pd.read_csv("Test_Dataset/Air_NRM/small_scale/query_NP_Flow.csv")
result_df = Batch_Process_Queries(df)
result_df.to_csv(LOG.dir / "answer_NP_Flow_smallscale_GPT4.1.csv", index=False)


# Small-scale NP execution is defined above; no CA fallback is present.


In [ ]:
# Small-scale NP execution is defined above; no CA fallback is present.


In [ ]:
# Small-scale NP execution is defined above; no CA fallback is present.


In [ ]:
# Small-scale NP execution is defined above; no CA fallback is present.


In [ ]:
# Small-scale NP execution is defined above; no CA fallback is present.
